In [ ]:
!pip install regex tqdm
!pip install diffusers transformers accelerate scipy
!pip install -U xformers
!pip install opencv-python
!pip install transformers

In [ ]:
import torch
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image, to_tensor

import PIL, cv2
from PIL import Image

from io import BytesIO
from IPython.display import display
import base64, json, requests
from matplotlib import pyplot as plt

import numpy as np
import copy
import sys

from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation


In [ ]:
from diffusers import StableDiffusionXLInpaintPipeline, EulerDiscreteScheduler

model_dir = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

scheduler = EulerDiscreteScheduler.from_pretrained(model_dir, subfolder="scheduler")

pipe = StableDiffusionXLInpaintPipeline.from_pretrained(model_dir,
                                                      scheduler=scheduler,
                                                      torch_dtype=torch.float16)

pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()

In [ ]:
target_width, target_height = 512, 512
source_image = Image.open("image 2.png").convert('RGB')

print(source_image.size)

width, height = source_image.size

source_image = source_image.crop((0, height-width, width, height))

source_image = source_image.resize((target_width, target_height), Image.LANCZOS)

print(source_image.size)

segmentation_image = np.asarray(source_image)
display(source_image)

In [ ]:
processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = SegformerForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to("cuda")

inputs = processor(images=segmentation_image, return_tensors="pt").to("cuda")

with torch.no_grad():
  outputs = model(**inputs)

logits = outputs.logits.cpu()
unsampled = torch.nn.functional.interpolate(
    logits, size = source_image.size[::-1],
    mode="bilinear", align_corners=False
)

pred = unsampled.argmax(dim=1)[0].numpy()

In [ ]:
import matplotlib.patches as mpatches

labels = {
    0: "Background", 1: "Hat", 2: "Hair", 3: "Sunglasses",
    4: "Upper-clothes", 5: "Skirt", 6: "Pants", 7: "Dress",
    8: "Belt", 9: "Left-shoe", 10: "Right-shoe", 11: "Face",
    12: "Left-leg", 13: "Right-leg", 14: "Left-arm", 15: "Right-arm",
    16: "Bag", 17: "Scarf"
}

# ---- Visualize all detected segments ----
unique_classes = np.unique(pred)
print("Detected segments:")
for cls in unique_classes:
    print(f"  [{cls}] {labels[cls]}")

# Plot original + segmentation map side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(source_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# color map for segmentation
seg_colored = plt.cm.tab20(pred / pred.max())
axes[1].imshow(seg_colored)
axes[1].set_title("Segmentation Map")
axes[1].axis("off")

# add legend for detected classes only
patches = [mpatches.Patch(color=plt.cm.tab20(cls / pred.max()),
           label=f"[{cls}] {labels[cls]}") for cls in unique_classes]
axes[1].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
def show_anns(anns):
  if len(anns) == 0:
    return

  sorted_anns = sorted(enumerate(anns), key=(lambda x: x[1]['area']), reverse=True)
  ax = plt.gca()

  ax.set_autoscale_on(False)

  for original_idx, ann in sorted_anns:
    m = ann['segmentation']
    img = np.ones((m.shape[0], m.shape[1], 3))

    color_mask = np.random.random((1, 3)).tolist()[0]

    for i in range(3):
      img[:,:,i] = color_mask[i]

    ax.imshow(np.dstack((img, m*0.35)))

    contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
      cnt = contours[0]
      M = cv2.moments(cnt)

      if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        ax.text(cx, cy, str(original_idx), color='white', fontsize=16, ha='center', va='center', fontweight='bold')

In [ ]:
def create_image_grid(original_image, images, names, rows, columns):
    names = copy.copy(names)  # Create a copy of the names list to avoid modifying the external variable
    images = copy.copy(images)  # Create a copy of the images list to avoid modifying the external variable

    # Check if images is a tensor
    if torch.is_tensor(images):
        # Check if the number of tensor images and names is equal
        assert images.size(0) == len(names), "Number of images and names should be equal"

        # Check if there are enough images for the specified grid size
        assert images.size(0) >= (rows * columns) - 1 - 1, "Not enough images for the specified grid size"

        # Convert tensor images to PIL images and apply sigmoid normalization
        images = [to_pil_image(torch.sigmoid(img)) for img in images]
    else:
        # Check if the number of PIL images and names is equal
        assert len(images) == len(names), "Number of images and names should be equal"

    # Check if there are enough images for the specified grid size
    assert len(images) >= (rows * columns) - 1 - 1, "Not enough images for the specified grid size"

    # Add the original image to the beginning of the images list
    images.insert(0, original_image)

    # Add an empty name for the original image to the beginning of the names list
    names.insert(0, '')

    # Create a figure with specified rows and columns
    fig, axes = plt.subplots(rows, columns, figsize=(15, 15))

    # Iterate through the images and names
    for idx, (img, name) in enumerate(zip(images, names)):
        # Calculate the row and column index for the current image
        row, col = divmod(idx, columns)

        # Add the image to the grid
        axes[row, col].imshow(img, cmap='gray' if idx > 0 and torch.is_tensor(images) else None)

        # Set the title (name) for the subplot
        axes[row, col].set_title(name)

        # Turn off axes for the subplot
        axes[row, col].axis('off')

    # Iterate through unused grid cells
    for idx in range(len(images), rows * columns):
        # Calculate the row and column index for the current cell
        row, col = divmod(idx, columns)

        # Turn off axes for the unused grid cell
        axes[row, col].axis('off')

    # Adjust the subplot positions to eliminate overlaps
    plt.tight_layout()

    # Display the grid of images with their names
    plt.show()


In [ ]:
mask_index = 2

selected_mask = (pred == mask_index).astype(np.uint8) * 255
stable_diffusion_mask = PIL.Image.fromarray(selected_mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(source_image)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(stable_diffusion_mask, cmap="gray")
axes[1].set_title(f"Mask: [{mask_index}] {labels[mask_index]}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"Extracting mask for: [{mask_index}] {labels[mask_index]}")

In [ ]:
prompts = [
    "blonde hair",
    "Pink hair"
]

generator = torch.Generator(device="cuda")

encoded_images = []

for i in range(len(prompts)):
  negative_prompt = "blurry edges, inconsistent lighting, artifacts, distorted, watermark"

  image = pipe(
      prompt=prompts[i],
      generator=generator,
      negative_prompt=negative_prompt,
      image=source_image,
      mask_image=stable_diffusion_mask
  ).images[0]

  encoded_images.append(image)

In [ ]:
create_image_grid(source_image, encoded_images, prompts, 2, 2)